# Clustering exploratoire -- Balance des paiements 2014

`methode.tex`, 1 annee (2014), 2 approches : (1) ACP + variables supplementaires, (2) clustering direct (5 methodes, sans ACP puis avec ACP).


---
## 1. Imports et configuration

In [25]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.figure_factory as ff
import plotly.express as px
from plotly.subplots import make_subplots

from sklearn.preprocessing import RobustScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans, AgglomerativeClustering, SpectralClustering
from sklearn.mixture import GaussianMixture
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score, adjusted_rand_score, silhouette_samples
from scipy.cluster.hierarchy import linkage, dendrogram, fcluster
from scipy.spatial.distance import cdist

RNG_SEED = 42
np.random.seed(RNG_SEED)


In [26]:
BASE = Path(os.path.abspath('')).parent

ANNEE_ETUDE = 2014
K_MIN, K_MAX = 2, 5
K_VALEURS = list(range(K_MIN, K_MAX + 1))

# Pays CEMAC (sous-ensemble de PAYS, bop_fmi.ipynb section 1) -- mis en evidence dans tous
# les graphiques et dans le positionnement final.
CEMAC = {
    'CMR': 'Cameroun', 'GAB': 'Gabon', 'COG': 'Republique du Congo',
    'TCD': 'Tchad', 'CAF': 'Republique centrafricaine', 'GNQ': 'Guinee equatoriale',
}


---
## 2. Chargement et preparation des donnees

In [27]:
chemin_clustering = BASE / 'data' / 'DONNEES_CLUSTERING.csv'
chemin_metriques = BASE / 'data' / 'METRIQUES_ANALYSE.csv'
chemin_correspondance = BASE / 'data' / 'CORRESPONDANCE_COLONNES_CLUSTERING.csv'

donnees = pd.read_csv(chemin_clustering)
donnees_annee = donnees[donnees['ANNEE'] == ANNEE_ETUDE].drop(columns=['ANNEE']).set_index('COUNTRY_CODE')

metriques = pd.read_csv(chemin_metriques)
metriques_annee = metriques[metriques['ANNEE'] == ANNEE_ETUDE].drop(columns=['ANNEE']).set_index('COUNTRY_CODE')

print(f"{ANNEE_ETUDE} : {len(donnees_annee)} pays dans DONNEES_CLUSTERING.csv, "
      f"{len(metriques_annee)} pays dans METRIQUES_ANALYSE.csv.")

if len(donnees_annee) < 15:
    print("ATTENTION : tres peu de pays pour cette annee -- verifier que bop_fmi.ipynb a bien "
          "ete re-execute (ANNEE_ETUDE_DEBUT = 2014, sections 5, 7-14, 14ter) avant de continuer.")

print(f"Colonnes actives ({donnees_annee.shape[1]}) : {sorted(donnees_annee.columns.tolist())}")


2014 : 20 pays dans DONNEES_CLUSTERING.csv, 57 pays dans METRIQUES_ANALYSE.csv.
Colonnes actives (905) : ['A_NFA_T__D1_F3', 'A_NFA_T__D1_F5A', 'A_NFA_T__D1_FL', 'A_NFA_T__D2_F3', 'A_NFA_T__D2_F5A', 'A_NFA_T__D2_FL', 'A_NFA_T__D3_F3', 'A_NFA_T__D3_F5A', 'A_NFA_T__D3_FL', 'A_NFA_T__D_F', 'A_NFA_T__D_F3', 'A_NFA_T__D_F5', 'A_NFA_T__D_F52B', 'A_NFA_T__D_F52B_S123', 'A_NFA_T__D_F5A', 'A_NFA_T__D_F5B', 'A_NFA_T__D_FL', 'A_NFA_T__FXR_F712', 'A_NFA_T__FXR_F71OTR', 'A_NFA_T__FXR_F72', 'A_NFA_T__FXR_OPT', 'A_NFA_T__F_F7', 'A_NFA_T__F_F7_S121', 'A_NFA_T__F_F7_S122', 'A_NFA_T__F_F7_S12R', 'A_NFA_T__F_F7_S13', 'A_NFA_T__F_F7_S1V', 'A_NFA_T__F_F7_S1X', 'A_NFA_T__F_F7_S1Z', 'A_NFA_T__O_F', 'A_NFA_T__O_F2', 'A_NFA_T__O_F221_S122', 'A_NFA_T__O_F2_S121', 'A_NFA_T__O_F2_S121_L', 'A_NFA_T__O_F2_S121_S', 'A_NFA_T__O_F2_S122', 'A_NFA_T__O_F2_S122_L', 'A_NFA_T__O_F2_S122_S', 'A_NFA_T__O_F2_S12R', 'A_NFA_T__O_F2_S12R_L', 'A_NFA_T__O_F2_S12R_S', 'A_NFA_T__O_F2_S13', 'A_NFA_T__O_F2_S13_L', 'A_NFA_T__O_F2_S13_S'

In [28]:
# Entrees brutes completes : aucun poste BDP ni aucun pays retire. Un poste non rapporte par un
# pays vaut simplement 0 (convention BPM6 deja utilisee dans tout le reste du pipeline,
# bop_fmi.ipynb section 15) -- pas un filtrage par seuil de couverture, qui create des
# suppressions arbitraires sans justification economique.
COLONNE_PIB = 'PIB_USD_MILLIONS'
taux_nan_avant = donnees_annee.isna().mean().mean()

pays_sans_pib = donnees_annee[COLONNE_PIB].isna() | (donnees_annee[COLONNE_PIB] <= 0)
if pays_sans_pib.any():
    print(f"{pays_sans_pib.sum()} pays exclus (PIB manquant ou nul en {ANNEE_ETUDE}, "
          f"necessaire pour exprimer les postes en % du PIB) : "
          f"{donnees_annee.index[pays_sans_pib].tolist()}")
donnees_annee = donnees_annee[~pays_sans_pib].fillna(0.0)

print(f"Jeu final : {len(donnees_annee)} pays x {donnees_annee.shape[1]} variables "
      f"(toutes les entrees BDP conservees ; taux de valeurs manquantes avant remplissage par 0 : "
      f"{taux_nan_avant:.1%}).")
print(f"Pays : {sorted(donnees_annee.index.tolist())}")


1 pays exclus (PIB manquant ou nul en 2014, necessaire pour exprimer les postes en % du PIB) : ['VEN']
Jeu final : 19 pays x 905 variables (toutes les entrees BDP conservees ; taux de valeurs manquantes avant remplissage par 0 : 33.0%).
Pays : ['AGO', 'ALB', 'ARG', 'AZE', 'BOL', 'CMR', 'COG', 'COL', 'ECU', 'GHA', 'ITA', 'KAZ', 'LBY', 'NLD', 'PER', 'ROU', 'RUS', 'SDN', 'TTO']


### Correction d'echelle

USA en outlier (postes BDP en centaines de milliards vs millions ailleurs). Fix : postes + PIB en % du PIB, indicateurs de niveau (`PPPPC`, `PPPGDP`, `NGDPRPPPPC`) en log, `RobustScaler`.


In [29]:
MACRO_RATIO = ['NGDP_RPCH', 'GGXCNL_NGDP', 'GGXONLB_NGDP', 'PCPIPCH', 'PCPIEPCH',
               'GGX_NGDP', 'GGR_NGDP', 'GGXWDG_NGDP', 'NID_NGDP']
MACRO_NIVEAU_ABSOLU = ['PPPPC', 'PPPGDP', 'NGDPRPPPPC']
COLONNE_PIB = 'PIB_USD_MILLIONS'

colonnes_entrees = [c for c in donnees_annee.columns
                    if c not in MACRO_RATIO + MACRO_NIVEAU_ABSOLU + [COLONNE_PIB]]

donnees_transformees = donnees_annee.copy()
donnees_transformees[colonnes_entrees] = (
    donnees_annee[colonnes_entrees].div(donnees_annee[COLONNE_PIB], axis=0) * 100
)
donnees_transformees[COLONNE_PIB] = np.log(donnees_annee[COLONNE_PIB])
for col in MACRO_NIVEAU_ABSOLU:
    if col in donnees_transformees.columns:
        donnees_transformees[col] = np.sign(donnees_annee[col]) * np.log1p(donnees_annee[col].abs())

print(f"{len(colonnes_entrees)} postes BDP convertis en % du PIB ; PIB et "
      f"{sum(c in donnees_transformees.columns for c in MACRO_NIVEAU_ABSOLU)} indicateurs de "
      f"niveau absolu log-transformes ; {len(MACRO_RATIO)} indicateurs deja en ratio inchanges.")
display(donnees_transformees.describe().T[['mean', 'std', 'min', 'max']].head(10))


892 postes BDP convertis en % du PIB ; PIB et 3 indicateurs de niveau absolu log-transformes ; 9 indicateurs deja en ratio inchanges.


,mean,std,min,max
A_NFA_T__D1_F3,0.000000,0.000000,0.000000,0.000000
A_NFA_T__D1_F5A,1.124543,3.426579,-0.135229,15.033551
A_NFA_T__D1_FL,0.381431,1.215141,-0.146465,5.254251
A_NFA_T__D2_F3,0.000000,0.000000,0.000000,0.000000
A_NFA_T__D2_F5A,0.000760,0.014543,-0.028129,0.053809
A_NFA_T__D2_FL,0.193437,1.853289,-3.848175,6.861674
A_NFA_T__D3_F3,0.000000,0.000000,0.000000,0.000000
A_NFA_T__D3_F5A,-0.000119,0.000615,-0.002636,0.000371
A_NFA_T__D3_FL,0.115274,0.582766,-0.263401,2.502186
A_NFA_T__D_F,1.915681,4.498992,-0.135229,19.138531


In [30]:
# RobustScaler (mediane / IQR) plutot que StandardScaler : plus resistant aux quelques valeurs
# encore extremes qui subsisteraient malgre la conversion en % du PIB / log ci-dessus (ex. un
# pays avec un poste ponctuel tres inhabituel une annee donnee).
scaler = RobustScaler()
X = scaler.fit_transform(donnees_transformees.values)
pays_index = donnees_transformees.index.tolist()

print(f"Matrice standardisee (robuste) : {X.shape[0]} pays x {X.shape[1]} variables.")


Matrice standardisee (robuste) : 19 pays x 905 variables.


---
## 3. Approche 1 -- ACP + variables supplementaires

Actives = `DONNEES_CLUSTERING.csv`. Supplementaires = metriques de `METRIQUES_ANALYSE.csv` (projetees, pas utilisees pour calculer les axes).


In [31]:
pca = PCA(n_components=min(10, X.shape[0] - 1, X.shape[1]), random_state=RNG_SEED)
scores = pca.fit_transform(X)

variance_expliquee = pca.explained_variance_ratio_
print("Variance expliquee par axe :")
for i, v in enumerate(variance_expliquee[:5], start=1):
    print(f"  Axe {i} : {v:.1%} (cumul {variance_expliquee[:i].sum():.1%})")


Variance expliquee par axe :
  Axe 1 : 95.9% (cumul 95.9%)
  Axe 2 : 2.9% (cumul 98.8%)
  Axe 3 : 0.8% (cumul 99.6%)
  Axe 4 : 0.2% (cumul 99.8%)
  Axe 5 : 0.1% (cumul 99.9%)


In [32]:
# Coordonnees des variables actives sur les 2 premiers axes (correlation variable/composante,
# convention ACP habituelle -- ex. FactoMineR) : loadings mis a l'echelle par sqrt(valeur propre).
loadings = pca.components_[:2].T * np.sqrt(pca.explained_variance_[:2])
df_loadings = pd.DataFrame(loadings, index=donnees_annee.columns, columns=['Axe1', 'Axe2'])

chemin_loadings = BASE / 'data' / f'ACP_LOADINGS_{ANNEE_ETUDE}.csv'
df_loadings.to_csv(chemin_loadings)
print(f"{len(df_loadings)} variables actives -> {chemin_loadings}")


905 variables actives -> m:\Travail\BEAC\balance_de_paiement\data\ACP_LOADINGS_2014.csv


In [33]:
# Variables supplementaires : les metriques deja calculees dans METRIQUES_ANALYSE.csv --
# reduites a des SOLDES (nets), jamais du credit/debit brut. CREDIT_BIENS/DEBIT_BIENS,
# CREDIT_SERVICES/DEBIT_SERVICES et CREDIT_REVENU_PRIMAIRE/DEBIT_REVENU_PRIMAIRE sont donc
# recombines en 3 soldes nets avant projection ; PIB_USD_MILLIONS est exclu car deja variable
# active (DONNEES_CLUSTERING.csv) -- pas de doublon actif/supplementaire.
metriques_supp = metriques_annee.copy()
metriques_supp['SOLDE_BIENS'] = metriques_supp['CREDIT_BIENS'] - metriques_supp['DEBIT_BIENS']
metriques_supp['SOLDE_SERVICES'] = metriques_supp['CREDIT_SERVICES'] - metriques_supp['DEBIT_SERVICES']
metriques_supp['SOLDE_REVENU_PRIMAIRE'] = metriques_supp['CREDIT_REVENU_PRIMAIRE'] - metriques_supp['DEBIT_REVENU_PRIMAIRE']
metriques_supp = metriques_supp.drop(columns=[
    'CREDIT_BIENS', 'DEBIT_BIENS', 'CREDIT_SERVICES', 'DEBIT_SERVICES',
    'CREDIT_REVENU_PRIMAIRE', 'DEBIT_REVENU_PRIMAIRE', 'PIB_USD_MILLIONS',
])

# Correlation entre chaque metrique (brute, non standardisee -- la correlation est invariante a
# l'echelle) et les scores factoriels des 2 premiers axes. Calcule uniquement sur les pays
# presents a la fois dans DONNEES_CLUSTERING.csv et METRIQUES_ANALYSE.csv.
pays_communs = [p for p in pays_index if p in metriques_supp.index]
if len(pays_communs) < len(pays_index):
    manquants = sorted(set(pays_index) - set(pays_communs))
    print(f"ATTENTION : {len(manquants)} pays absents de METRIQUES_ANALYSE.csv pour {ANNEE_ETUDE} "
          f"-- variables supplementaires non projetees pour ces pays : {manquants}")

scores_df = pd.DataFrame(scores[:, :2], index=pays_index, columns=['Axe1', 'Axe2'])

coord_supp = {}
for col in metriques_supp.columns:
    valeurs = metriques_supp.loc[pays_communs, col]
    if valeurs.notna().sum() < 3:
        continue
    valeurs_remplies = valeurs.fillna(valeurs.mean())
    s = scores_df.loc[pays_communs]
    coord_supp[col] = (
        np.corrcoef(valeurs_remplies, s['Axe1'])[0, 1],
        np.corrcoef(valeurs_remplies, s['Axe2'])[0, 1],
    )

df_supp = pd.DataFrame(coord_supp, index=['Axe1', 'Axe2']).T
chemin_supp = BASE / 'data' / f'ACP_VARIABLES_SUPPLEMENTAIRES_{ANNEE_ETUDE}.csv'
df_supp.to_csv(chemin_supp)
print(f"{len(df_supp)} variables supplementaires (metriques, soldes nets uniquement) -> {chemin_supp}")
display(df_supp)


16 variables supplementaires (metriques, soldes nets uniquement) -> m:\Travail\BEAC\balance_de_paiement\data\ACP_VARIABLES_SUPPLEMENTAIRES_2014.csv


,Axe1,Axe2
SOLDE_COURANT,-0.044711,0.748005
IDE_NET,-0.040064,0.931755
EO_NET,-0.073238,0.122156
TC_TAUX_COUVERTURE_BIENS,-0.005537,-0.029923
TM_TAUX_COUVERTURE_IMPORTATIONS,-0.062368,-0.093832
TAS_TAUX_ABSORPTION_SERVICES,-0.072448,-0.176744
TFR_TAUX_FUITE_REVENU,-0.037094,0.500309
TRE_TAUX_RETENTION,0.046083,0.065481
RATIO_IDE_SOLDE_COURANT,-0.086466,0.268185
RATIO_ERREURS_OMISSIONS,-0.110020,-0.169130


In [34]:
# Biplot : pays (points) + variables actives les plus contributives (fleches grises, les 49
# variables completes rendraient le graphique illisible -- top 15 par contribution aux 2 premiers
# axes, table complete deja sauvegardee ci-dessus) + variables supplementaires (fleches oranges).
libelles_courts = {c: c for c in donnees_annee.columns}
if chemin_correspondance.exists():
    corr_df = pd.read_csv(chemin_correspondance).set_index('CODE_COLONNE')['LIBELLE_COMPLET']
    for c in donnees_annee.columns:
        if c in corr_df.index:
            libelles_courts[c] = str(corr_df[c]).split(' | ')[-1][:30]

contribution = (df_loadings ** 2).sum(axis=1)
top_variables = contribution.sort_values(ascending=False).head(15).index

portee_scores = scores_df[['Axe1', 'Axe2']].abs().values.max()
echelle_actives = 0.8 * portee_scores / df_loadings.loc[top_variables].abs().values.max()
echelle_supp = 0.8 * portee_scores

fig = go.Figure()

# Points pays (CEMAC en rouge, le reste en bleu)
for est_cemac, couleur, nom_legende in [(True, 'crimson', 'CEMAC'), (False, 'steelblue', 'Autres pays')]:
    sous_ensemble = [p for p in pays_index if (p in CEMAC) == est_cemac]
    fig.add_trace(go.Scatter(
        x=scores_df.loc[sous_ensemble, 'Axe1'], y=scores_df.loc[sous_ensemble, 'Axe2'],
        mode='markers+text', text=sous_ensemble, textposition='top center',
        textfont=dict(size=9, color=couleur),
        marker=dict(size=9, color=couleur), name=nom_legende,
    ))

# Fleches variables actives (grises) et supplementaires (oranges), via annotations flechees
for var in top_variables:
    a1, a2 = df_loadings.loc[var]
    fig.add_annotation(
        x=a1 * echelle_actives, y=a2 * echelle_actives, ax=0, ay=0,
        xref='x', yref='y', axref='x', ayref='y',
        showarrow=True, arrowhead=2, arrowwidth=1, arrowcolor='grey', opacity=0.7,
    )
    fig.add_annotation(
        x=a1 * echelle_actives, y=a2 * echelle_actives, text=libelles_courts[var],
        showarrow=False, font=dict(size=9, color='dimgrey'), xanchor='left',
    )

for var, (a1, a2) in df_supp.iterrows():
    fig.add_annotation(
        x=a1 * echelle_supp, y=a2 * echelle_supp, ax=0, ay=0,
        xref='x', yref='y', axref='x', ayref='y',
        showarrow=True, arrowhead=2, arrowwidth=2, arrowcolor='darkorange',
    )
    fig.add_annotation(
        x=a1 * echelle_supp, y=a2 * echelle_supp, text=var,
        showarrow=False, font=dict(size=10, color='darkorange'), xanchor='left',
    )

fig.add_hline(y=0, line=dict(color='grey', width=0.5))
fig.add_vline(x=0, line=dict(color='grey', width=0.5))
fig.update_layout(
    title=(f"ACP -- Balance des paiements {ANNEE_ETUDE} (CEMAC en rouge)<br>"
           "<sup>Fleches grises = 15 variables actives les plus contributives, "
           "fleches oranges = metriques supplementaires</sup>"),
    xaxis_title=f"Axe 1 ({variance_expliquee[0]:.1%})",
    yaxis_title=f"Axe 2 ({variance_expliquee[1]:.1%})",
    width=1000, height=850, template='plotly_white',
)

chemin_fig = BASE / 'figures' / f'acp_biplot_{ANNEE_ETUDE}.png'
chemin_fig.parent.mkdir(parents=True, exist_ok=True)
fig.write_image(chemin_fig, scale=2)
print(f"Biplot sauvegarde -> {chemin_fig}")
fig.show()


Biplot sauvegarde -> m:\Travail\BEAC\balance_de_paiement\figures\acp_biplot_2014.png


---
## 4. Approche 2 -- clustering sans ACP

5 methodes (HAC-Ward, k-means++, k-medoids, GMM, Spectral) x k in {2..5}, valides par Silhouette/Davies-Bouldin/Calinski-Harabasz. GMM en covariance diagonale (p proche de n). k-medoids : implementation maison (pas de scikit-learn-extra).


In [35]:
def kmedoids_pam(X, k, seed=RNG_SEED, max_iter=100):
    """PAM simplifie (variante 'Alternate') : pas de dependance a scikit-learn-extra."""
    rng = np.random.default_rng(seed)
    n = X.shape[0]
    medoid_idx = rng.choice(n, k, replace=False)
    for _ in range(max_iter):
        dist = cdist(X, X[medoid_idx])
        labels = dist.argmin(axis=1)
        nouveaux_medoides = medoid_idx.copy()
        for j in range(k):
            membres = np.where(labels == j)[0]
            if len(membres) == 0:
                continue
            dist_intra = cdist(X[membres], X[membres]).sum(axis=1)
            nouveaux_medoides[j] = membres[dist_intra.argmin()]
        if np.array_equal(nouveaux_medoides, medoid_idx):
            break
        medoid_idx = nouveaux_medoides
    dist = cdist(X, X[medoid_idx])
    return dist.argmin(axis=1)


def labels_hac_ward(X, k):
    return AgglomerativeClustering(n_clusters=k, linkage='ward').fit_predict(X)


def labels_kmeans(X, k):
    return KMeans(n_clusters=k, init='k-means++', n_init=10, random_state=RNG_SEED).fit_predict(X)


def labels_kmedoids(X, k):
    return kmedoids_pam(X, k)


def labels_gmm(X, k):
    gmm = GaussianMixture(n_components=k, covariance_type='diag', reg_covar=1e-2,
                           random_state=RNG_SEED, n_init=5)
    return gmm.fit_predict(X)


def labels_spectral(X, k):
    return SpectralClustering(n_clusters=k, affinity='rbf', random_state=RNG_SEED,
                               assign_labels='kmeans').fit_predict(X)


METHODES = {
    'HAC-Ward': labels_hac_ward,
    'k-means++': labels_kmeans,
    'k-medoids': labels_kmedoids,
    'GMM': labels_gmm,
    'Spectral': labels_spectral,
}
print(f"{len(METHODES)} methodes prêtes : {list(METHODES)}")


5 methodes prêtes : ['HAC-Ward', 'k-means++', 'k-medoids', 'GMM', 'Spectral']


In [36]:
resultats_validation = []
for nom_methode, fonction in METHODES.items():
    for k in K_VALEURS:
        try:
            labels = fonction(X, k)
            if len(set(labels)) < 2:
                print(f"IGNORE {nom_methode} k={k} : un seul cluster non vide produit.")
                continue
            resultats_validation.append({
                'methode': nom_methode, 'k': k,
                'silhouette': silhouette_score(X, labels),
                'davies_bouldin': davies_bouldin_score(X, labels),
                'calinski_harabasz': calinski_harabasz_score(X, labels),
            })
        except Exception as e:
            print(f"ECHEC {nom_methode} k={k} : {e}")

df_validation = pd.DataFrame(resultats_validation)
chemin_validation = BASE / 'data' / f'CLUSTERING_VALIDATION_{ANNEE_ETUDE}.csv'
df_validation.to_csv(chemin_validation, index=False)
print(f"{len(df_validation)} configurations (methode x k) evaluees -> {chemin_validation}")
display(df_validation.sort_values(['methode', 'k']))


20 configurations (methode x k) evaluees -> m:\Travail\BEAC\balance_de_paiement\data\CLUSTERING_VALIDATION_2014.csv


c:\Users\Miguel\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\manifold\_spectral_embedding.py:273: UserWarning: Graph is not fully connected, spectral embedding may not work as expected.
  warnings.warn(
c:\Users\Miguel\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\manifold\_spectral_embedding.py:273: UserWarning: Graph is not fully connected, spectral embedding may not work as expected.
  warnings.warn(
c:\Users\Miguel\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\manifold\_spectral_embedding.py:273: UserWarning: Graph is not fully connected, spectral embedding may not work as expected.
  warnings.warn(
c:\Users\Miguel\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\manifold\_spectral_embedding.py:273: UserWarning: Graph is not fully connected, spectral embedding may not work as expected.
  warnings.warn(


,methode,k,silhouette,davies_bouldin,calinski_harabasz
12,GMM,2,0.274794,0.673164,0.510807
13,GMM,3,0.510812,0.231604,71.878555
14,GMM,4,0.623126,0.304802,128.820906
15,GMM,5,0.623605,0.261218,222.348388
0,HAC-Ward,2,0.857569,0.060600,115.719541
1,HAC-Ward,3,0.626840,0.368123,115.258762
2,HAC-Ward,4,0.623126,0.304802,128.820906
3,HAC-Ward,5,0.623605,0.261218,222.348388
16,Spectral,2,-0.234113,2.090654,0.566219
17,Spectral,3,-0.188604,3.521814,0.500160


In [37]:
# Score composite : rang moyen sur les 3 indices (silhouette et calinski-harabasz a maximiser,
# davies-bouldin a minimiser) -- pas de ponderation a priori entre les 3, comme convenu.
df_rangs = df_validation.copy()
df_rangs['rang_silhouette'] = df_rangs['silhouette'].rank(ascending=False)
df_rangs['rang_db'] = df_rangs['davies_bouldin'].rank(ascending=True)
df_rangs['rang_ch'] = df_rangs['calinski_harabasz'].rank(ascending=False)
df_rangs['rang_moyen'] = df_rangs[['rang_silhouette', 'rang_db', 'rang_ch']].mean(axis=1)

meilleure_config = df_rangs.sort_values('rang_moyen').iloc[0]
print(f"Meilleure configuration (rang moyen sur les 3 indices) : "
      f"{meilleure_config['methode']}, k={int(meilleure_config['k'])}")
display(df_rangs.sort_values('rang_moyen').head(10))


Meilleure configuration (rang moyen sur les 3 indices) : HAC-Ward, k=2


,methode,k,silhouette,davies_bouldin,calinski_harabasz,rang_silhouette,rang_db,rang_ch,rang_moyen
0,HAC-Ward,2,0.857569,0.060600,115.719541,1.5,1.5,7.5,3.500000
4,k-means++,2,0.857569,0.060600,115.719541,1.5,1.5,7.5,3.500000
3,HAC-Ward,5,0.623605,0.261218,222.348388,6.0,6.0,2.0,4.666667
7,k-means++,5,0.623605,0.261218,222.348388,6.0,6.0,2.0,4.666667
15,GMM,5,0.623605,0.261218,222.348388,6.0,6.0,2.0,4.666667
2,HAC-Ward,4,0.623126,0.304802,128.820906,9.0,9.0,5.0,7.666667
6,k-means++,4,0.623126,0.304802,128.820906,9.0,9.0,5.0,7.666667
14,GMM,4,0.623126,0.304802,128.820906,9.0,9.0,5.0,7.666667
1,HAC-Ward,3,0.626840,0.368123,115.258762,3.5,11.5,9.5,8.166667
5,k-means++,3,0.626840,0.368123,115.258762,3.5,11.5,9.5,8.166667


### Separation par methode

Meilleur k propre a chaque methode. Projection ACP coloree par cluster, diagramme de silhouette, comparaison des indices.


In [38]:
meilleurs_par_methode = (
    df_rangs.loc[df_rangs.groupby('methode')['rang_moyen'].idxmin()]
    .set_index('methode')[['k', 'silhouette', 'davies_bouldin', 'calinski_harabasz', 'rang_moyen']]
    .sort_values('rang_moyen')
)
display(meilleurs_par_methode)

labels_par_methode = {
    methode: METHODES[methode](X, int(ligne['k']))
    for methode, ligne in meilleurs_par_methode.iterrows()
}


,k,silhouette,davies_bouldin,calinski_harabasz,rang_moyen
methode,,,,,
HAC-Ward,2,0.857569,0.060600,115.719541,3.500000
k-means++,2,0.857569,0.060600,115.719541,3.500000
GMM,5,0.623605,0.261218,222.348388,4.666667
k-medoids,4,0.521525,0.215402,64.531247,9.000000
Spectral,2,-0.234113,2.090654,0.566219,16.000000


c:\Users\Miguel\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\manifold\_spectral_embedding.py:273: UserWarning: Graph is not fully connected, spectral embedding may not work as expected.
  warnings.warn(


In [39]:
palette = px.colors.qualitative.Set2
methodes_ordre = meilleurs_par_methode.index.tolist()
n_cols = 3
n_rows = -(-len(methodes_ordre) // n_cols)

fig = make_subplots(
    rows=n_rows, cols=n_cols,
    subplot_titles=[f"{m} (k={int(meilleurs_par_methode.loc[m, 'k'])})" for m in methodes_ordre],
)

for idx_m, methode in enumerate(methodes_ordre):
    row, col = idx_m // n_cols + 1, idx_m % n_cols + 1
    labels = labels_par_methode[methode]
    tailles = pd.Series(labels).value_counts().sort_index()
    print(f"{methode} : effectifs par cluster {tailles.to_dict()}")
    for cluster_id in sorted(set(labels)):
        membres = [p for p, l in zip(pays_index, labels) if l == cluster_id]
        fig.add_trace(go.Scatter(
            x=scores_df.loc[membres, 'Axe1'], y=scores_df.loc[membres, 'Axe2'],
            mode='markers+text', text=membres, textposition='top center',
            textfont=dict(size=7), showlegend=False,
            marker=dict(
                size=[13 if p in CEMAC else 8 for p in membres],
                symbol=['star' if p in CEMAC else 'circle' for p in membres],
                color=palette[cluster_id % len(palette)],
                line=dict(width=1, color='black'),
            ),
        ), row=row, col=col)

fig.update_layout(
    height=420 * n_rows, width=440 * n_cols, template='plotly_white',
    title_text=f"Separation des clusters par methode -- projection ACP axes 1-2, {ANNEE_ETUDE}",
)

chemin_sep = BASE / 'figures' / f'clusters_separation_pca_{ANNEE_ETUDE}.png'
fig.write_image(chemin_sep, scale=2)
print(f"Grille de separation (projection ACP) sauvegardee -> {chemin_sep}")
fig.show()


HAC-Ward : effectifs par cluster {0: 18, 1: 1}
k-means++ : effectifs par cluster {0: 18, 1: 1}
GMM : effectifs par cluster {0: 14, 1: 1, 2: 2, 3: 1, 4: 1}
k-medoids : effectifs par cluster {0: 1, 1: 16, 2: 1, 3: 1}
Spectral : effectifs par cluster {0: 18, 1: 1}
Grille de separation (projection ACP) sauvegardee -> m:\Travail\BEAC\balance_de_paiement\figures\clusters_separation_pca_2014.png


In [40]:
fig_sil = make_subplots(
    rows=n_rows, cols=n_cols,
    subplot_titles=[f"{m} (k={int(meilleurs_par_methode.loc[m, 'k'])})" for m in methodes_ordre],
)

for idx, methode in enumerate(methodes_ordre):
    row, col = idx // n_cols + 1, idx % n_cols + 1
    labels = labels_par_methode[methode]
    valeurs_sil = silhouette_samples(X, labels)
    score_moyen = valeurs_sil.mean()

    y_offset = 0
    for cluster_id in sorted(set(labels)):
        valeurs_cluster = np.sort(valeurs_sil[labels == cluster_id])
        n_cluster = len(valeurs_cluster)
        fig_sil.add_trace(go.Bar(
            x=valeurs_cluster, y=list(range(y_offset, y_offset + n_cluster)),
            orientation='h', marker=dict(color=palette[cluster_id % len(palette)]),
            showlegend=False, width=0.9,
        ), row=row, col=col)
        y_offset += n_cluster + 2

    fig_sil.add_vline(x=score_moyen, line=dict(color='red', dash='dash'), row=row, col=col)

fig_sil.update_layout(
    height=380 * n_rows, width=420 * n_cols, template='plotly_white',
    title_text=f"Diagrammes de silhouette par methode, {ANNEE_ETUDE} (ligne rouge = silhouette moyenne)",
)
fig_sil.update_xaxes(range=[-1, 1])

chemin_sil = BASE / 'figures' / f'silhouette_par_methode_{ANNEE_ETUDE}.png'
fig_sil.write_image(chemin_sil, scale=2)
print(f"Diagrammes de silhouette sauvegardes -> {chemin_sil}")
fig_sil.show()


Diagrammes de silhouette sauvegardes -> m:\Travail\BEAC\balance_de_paiement\figures\silhouette_par_methode_2014.png


In [41]:
fig_comp = make_subplots(
    rows=1, cols=3,
    subplot_titles=['Silhouette (a maximiser)', 'Davies-Bouldin (a minimiser)', 'Calinski-Harabasz (a maximiser)'],
)
fig_comp.add_trace(go.Bar(x=methodes_ordre, y=meilleurs_par_methode['silhouette'],
                           marker_color='steelblue', showlegend=False), row=1, col=1)
fig_comp.add_trace(go.Bar(x=methodes_ordre, y=meilleurs_par_methode['davies_bouldin'],
                           marker_color='indianred', showlegend=False), row=1, col=2)
fig_comp.add_trace(go.Bar(x=methodes_ordre, y=meilleurs_par_methode['calinski_harabasz'],
                           marker_color='seagreen', showlegend=False), row=1, col=3)

fig_comp.update_layout(
    title_text=f"Comparaison des indices de validation par methode (a son propre meilleur k), {ANNEE_ETUDE}",
    template='plotly_white', height=420, width=1150,
)

chemin_comp = BASE / 'figures' / f'comparaison_indices_{ANNEE_ETUDE}.png'
fig_comp.write_image(chemin_comp, scale=2)
print(f"Comparaison des indices sauvegardee -> {chemin_comp}")
fig_comp.show()


Comparaison des indices sauvegardee -> m:\Travail\BEAC\balance_de_paiement\figures\comparaison_indices_2014.png


In [42]:
Z = linkage(X, method='ward')

fig = ff.create_dendrogram(
    X, labels=pays_index, linkagefun=lambda _: Z, orientation='bottom',
)
for trace in fig['data']:
    trace['line']['color'] = '#555555'

# Les libelles de feuilles sont portes par xaxis.ticktext (pas des annotations) -- Plotly
# accepte un sous-ensemble de balises HTML (b, span/color) dans le texte des ticks pour les
# mettre en evidence individuellement.
ticktext = [f"<b><span style='color:crimson'>{lbl}</span></b>" if lbl in CEMAC else lbl
            for lbl in fig.layout.xaxis.ticktext]
fig.update_xaxes(ticktext=ticktext, tickfont=dict(size=9))

fig.update_layout(
    title=f"Dendrogramme HAC-Ward -- Balance des paiements {ANNEE_ETUDE} (CEMAC en rouge)",
    width=1400, height=550, template='plotly_white',
)

chemin_dendro = BASE / 'figures' / f'dendrogramme_ward_{ANNEE_ETUDE}.png'
fig.write_image(chemin_dendro, scale=2)
print(f"Dendrogramme sauvegarde -> {chemin_dendro}")
fig.show()


Dendrogramme sauvegarde -> m:\Travail\BEAC\balance_de_paiement\figures\dendrogramme_ward_2014.png


### Hybride HAC-Ward -> k-means++

Ward fixe k et initialise les centroides, k-means++ raffine.


In [43]:
k_retenu = int(meilleure_config['k'])
labels_ward = fcluster(Z, t=k_retenu, criterion='maxclust') - 1

centroides_ward = np.array([X[labels_ward == j].mean(axis=0) for j in range(k_retenu)])
labels_hybride = KMeans(n_clusters=k_retenu, init=centroides_ward, n_init=1,
                         random_state=RNG_SEED).fit_predict(X)

ari = adjusted_rand_score(labels_ward, labels_hybride)
silhouette_ward = silhouette_score(X, labels_ward)
silhouette_hybride = silhouette_score(X, labels_hybride)

print(f"Accord Ward vs hybride Ward->k-means++ (k={k_retenu}) : ARI = {ari:.3f} "
      f"(1 = partitions identiques, 0 = independance)")
print(f"Silhouette Ward seul : {silhouette_ward:.3f}  |  Silhouette hybride : {silhouette_hybride:.3f}")


Accord Ward vs hybride Ward->k-means++ (k=2) : ARI = 1.000 (1 = partitions identiques, 0 = independance)
Silhouette Ward seul : 0.858  |  Silhouette hybride : 0.858


---
## 5. Assignation finale et positionnement des pays CEMAC

In [44]:
# Retient la partition (meilleure config seule, ou hybride si elle fait mieux en silhouette)
# avec le plus grand indice de silhouette.
if silhouette_hybride >= max(silhouette_ward, meilleure_config['silhouette']):
    labels_finaux, methode_finale = labels_hybride, 'Hybride Ward->k-means++'
elif silhouette_ward >= meilleure_config['silhouette']:
    labels_finaux, methode_finale = labels_ward, 'HAC-Ward'
else:
    labels_finaux, methode_finale = METHODES[meilleure_config['methode']](X, k_retenu), meilleure_config['methode']

df_assignation = pd.DataFrame({'COUNTRY_CODE': pays_index, 'CLUSTER': labels_finaux})
df_assignation['PAYS_CEMAC'] = df_assignation['COUNTRY_CODE'].isin(CEMAC)

chemin_assignation = BASE / 'data' / f'CLUSTERING_ASSIGNATION_{ANNEE_ETUDE}.csv'
df_assignation.to_csv(chemin_assignation, index=False)

print(f"Methode retenue : {methode_finale}, k={k_retenu} -> {chemin_assignation}")
print(f"Taille des clusters : {df_assignation['CLUSTER'].value_counts().sort_index().to_dict()}")

print("\nPositionnement des pays CEMAC :")
for code_pays, nom in CEMAC.items():
    ligne = df_assignation[df_assignation['COUNTRY_CODE'] == code_pays]
    if ligne.empty:
        print(f"  {nom} ({code_pays}) : absent du jeu de donnees {ANNEE_ETUDE}")
        continue
    cluster = ligne['CLUSTER'].iloc[0]
    pairs = df_assignation[
        (df_assignation['CLUSTER'] == cluster) & (~df_assignation['PAYS_CEMAC'])
    ]['COUNTRY_CODE'].tolist()
    print(f"  {nom} ({code_pays}) -> cluster {cluster}, partage avec : {pairs}")

display(df_assignation.sort_values(['CLUSTER', 'COUNTRY_CODE']))


Methode retenue : Hybride Ward->k-means++, k=2 -> m:\Travail\BEAC\balance_de_paiement\data\CLUSTERING_ASSIGNATION_2014.csv
Taille des clusters : {0: 18, 1: 1}

Positionnement des pays CEMAC :
  Cameroun (CMR) -> cluster 0, partage avec : ['AGO', 'ALB', 'ARG', 'AZE', 'BOL', 'COL', 'GHA', 'ITA', 'KAZ', 'LBY', 'NLD', 'PER', 'ROU', 'RUS', 'SDN', 'TTO']
  Gabon (GAB) : absent du jeu de donnees 2014
  Republique du Congo (COG) -> cluster 0, partage avec : ['AGO', 'ALB', 'ARG', 'AZE', 'BOL', 'COL', 'GHA', 'ITA', 'KAZ', 'LBY', 'NLD', 'PER', 'ROU', 'RUS', 'SDN', 'TTO']
  Tchad (TCD) : absent du jeu de donnees 2014
  Republique centrafricaine (CAF) : absent du jeu de donnees 2014
  Guinee equatoriale (GNQ) : absent du jeu de donnees 2014


,COUNTRY_CODE,CLUSTER,PAYS_CEMAC
0,AGO,0,False
1,ALB,0,False
2,ARG,0,False
3,AZE,0,False
4,BOL,0,False
5,CMR,0,True
6,COG,0,True
7,COL,0,False
9,GHA,0,False
10,ITA,0,False


---
## 6. Test avec ACP

`methode.tex` : seuil de variance expliquee (70-95%) comme hyperparametre. Compare a la branche "sans ACP", garde la meilleure.


In [45]:
VARIANCE_SEUILS = [0.70, 0.80, 0.85, 0.90, 0.95]

resultats_avec_acp = []
composantes_par_seuil = {}
for seuil in VARIANCE_SEUILS:
    pca_seuil = PCA(n_components=seuil, svd_solver='full', random_state=RNG_SEED)
    X_reduit = pca_seuil.fit_transform(X)
    composantes_par_seuil[seuil] = X_reduit.shape[1]

    for nom_methode, fonction in METHODES.items():
        for k in K_VALEURS:
            try:
                labels = fonction(X_reduit, k)
                if len(set(labels)) < 2:
                    continue
                resultats_avec_acp.append({
                    'seuil_variance': seuil, 'n_composantes': X_reduit.shape[1],
                    'methode': nom_methode, 'k': k,
                    'silhouette': silhouette_score(X_reduit, labels),
                    'davies_bouldin': davies_bouldin_score(X_reduit, labels),
                    'calinski_harabasz': calinski_harabasz_score(X_reduit, labels),
                })
            except Exception as e:
                print(f"ECHEC seuil={seuil:.0%} {nom_methode} k={k} : {e}")

for seuil, n_comp in composantes_par_seuil.items():
    print(f"Seuil {seuil:.0%} -> {n_comp} composantes retenues (sur {X.shape[1]} variables)")

df_avec_acp = pd.DataFrame(resultats_avec_acp)
chemin_validation_acp = BASE / 'data' / f'CLUSTERING_VALIDATION_ACP_{ANNEE_ETUDE}.csv'
df_avec_acp.to_csv(chemin_validation_acp, index=False)
print(f"{len(df_avec_acp)} configurations (seuil x methode x k) evaluees -> {chemin_validation_acp}")


c:\Users\Miguel\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\manifold\_spectral_embedding.py:273: UserWarning: Graph is not fully connected, spectral embedding may not work as expected.
  warnings.warn(
c:\Users\Miguel\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\manifold\_spectral_embedding.py:273: UserWarning: Graph is not fully connected, spectral embedding may not work as expected.
  warnings.warn(
c:\Users\Miguel\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\manifold\_spectral_embedding.py:273: UserWarning: Graph is not fully connected, spectral embedding may not work as expected.
  warnings.warn(
c:\Users\Miguel\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\manifold\_spectral_embedding.py:273: UserWarning: Graph is not fully connected, spectral embedding may not work as expected.
  warnings.warn(
c:\Users\Miguel\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\manifold\_spectral_emb

Seuil 70% -> 1 composantes retenues (sur 905 variables)
Seuil 80% -> 1 composantes retenues (sur 905 variables)
Seuil 85% -> 1 composantes retenues (sur 905 variables)
Seuil 90% -> 1 composantes retenues (sur 905 variables)
Seuil 95% -> 1 composantes retenues (sur 905 variables)
100 configurations (seuil x methode x k) evaluees -> m:\Travail\BEAC\balance_de_paiement\data\CLUSTERING_VALIDATION_ACP_2014.csv


c:\Users\Miguel\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\manifold\_spectral_embedding.py:273: UserWarning: Graph is not fully connected, spectral embedding may not work as expected.
  warnings.warn(
c:\Users\Miguel\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\manifold\_spectral_embedding.py:273: UserWarning: Graph is not fully connected, spectral embedding may not work as expected.
  warnings.warn(


In [46]:
def nommer_configuration(r):
    if pd.isna(r['seuil_variance']):
        branche = 'sans ACP'
    else:
        branche = f"ACP {r['seuil_variance']:.0%}"
    return f"{r['methode']} k={int(r['k'])} ({branche})"


df_sans_acp = df_validation.copy()
df_sans_acp['seuil_variance'] = np.nan
df_sans_acp['n_composantes'] = X.shape[1]

df_global = pd.concat([df_sans_acp, df_avec_acp], ignore_index=True)
df_global['rang_silhouette'] = df_global['silhouette'].rank(ascending=False)
df_global['rang_db'] = df_global['davies_bouldin'].rank(ascending=True)
df_global['rang_ch'] = df_global['calinski_harabasz'].rank(ascending=False)
df_global['rang_moyen'] = df_global[['rang_silhouette', 'rang_db', 'rang_ch']].mean(axis=1)
df_global['configuration'] = df_global.apply(nommer_configuration, axis=1)

chemin_global = BASE / 'data' / f'CLUSTERING_VALIDATION_GLOBALE_{ANNEE_ETUDE}.csv'
df_global.to_csv(chemin_global, index=False)

meilleure_globale = df_global.sort_values('rang_moyen').iloc[0]
print("Top 10 configurations, toutes branches confondues (avec et sans ACP) :")
display(df_global.sort_values('rang_moyen')[
    ['configuration', 'silhouette', 'davies_bouldin', 'calinski_harabasz', 'rang_moyen']
].head(10))
print(f"\nMeilleure configuration globale -> {chemin_global} : "
      f"{meilleure_globale['configuration']} (rang moyen {meilleure_globale['rang_moyen']:.1f})")


Top 10 configurations, toutes branches confondues (avec et sans ACP) :


,configuration,silhouette,davies_bouldin,calinski_harabasz,rang_moyen
107,k-means++ k=5 (ACP 95%),0.741062,0.027518,2208.180339,24.5
95,GMM k=5 (ACP 90%),0.741062,0.027518,2208.180339,24.5
47,k-means++ k=5 (ACP 80%),0.741062,0.027518,2208.180339,24.5
67,k-means++ k=5 (ACP 85%),0.741062,0.027518,2208.180339,24.5
87,k-means++ k=5 (ACP 90%),0.741062,0.027518,2208.180339,24.5
55,GMM k=5 (ACP 80%),0.741062,0.027518,2208.180339,24.5
75,GMM k=5 (ACP 85%),0.741062,0.027518,2208.180339,24.5
115,GMM k=5 (ACP 95%),0.741062,0.027518,2208.180339,24.5
35,GMM k=5 (ACP 70%),0.741062,0.027518,2208.180339,24.5
27,k-means++ k=5 (ACP 70%),0.741062,0.027518,2208.180339,24.5



Meilleure configuration globale -> m:\Travail\BEAC\balance_de_paiement\data\CLUSTERING_VALIDATION_GLOBALE_2014.csv : k-means++ k=5 (ACP 95%) (rang moyen 24.5)


In [47]:
df_global_branche = df_global.copy()
df_global_branche['branche'] = df_global_branche['seuil_variance'].apply(
    lambda s: 'Sans ACP' if pd.isna(s) else f"ACP {s:.0%}"
)
meilleur_par_branche = (
    df_global_branche.loc[df_global_branche.groupby('branche')['rang_moyen'].idxmin()]
    .sort_values('rang_moyen')
)
display(meilleur_par_branche[
    ['branche', 'methode', 'k', 'silhouette', 'davies_bouldin', 'calinski_harabasz', 'rang_moyen']
])

fig_seuils = go.Figure(go.Bar(
    x=meilleur_par_branche['branche'], y=meilleur_par_branche['silhouette'],
    marker_color=['seagreen' if i == 0 else 'steelblue' for i in range(len(meilleur_par_branche))],
    text=[f"{m} k={int(k)}" for m, k in zip(meilleur_par_branche['methode'], meilleur_par_branche['k'])],
    textposition='outside',
))
fig_seuils.update_layout(
    title=f"Meilleur silhouette par branche (avec/sans ACP), {ANNEE_ETUDE}<br>"
          "<sup>Vert = branche retenue (meilleur rang moyen sur les 3 indices)</sup>",
    yaxis_title='Silhouette (meilleure config de la branche)', template='plotly_white',
    height=450, width=900,
)

chemin_seuils = BASE / 'figures' / f'comparaison_branches_acp_{ANNEE_ETUDE}.png'
fig_seuils.write_image(chemin_seuils, scale=2)
print(f"Comparaison des branches sauvegardee -> {chemin_seuils}")
fig_seuils.show()


,branche,methode,k,silhouette,davies_bouldin,calinski_harabasz,rang_moyen
27,ACP 70%,k-means++,5,0.741062,0.027518,2208.180339,24.500000
47,ACP 80%,k-means++,5,0.741062,0.027518,2208.180339,24.500000
67,ACP 85%,k-means++,5,0.741062,0.027518,2208.180339,24.500000
87,ACP 90%,k-means++,5,0.741062,0.027518,2208.180339,24.500000
107,ACP 95%,k-means++,5,0.741062,0.027518,2208.180339,24.500000
0,Sans ACP,HAC-Ward,2,0.857569,0.060600,115.719541,46.833333


Comparaison des branches sauvegardee -> m:\Travail\BEAC\balance_de_paiement\figures\comparaison_branches_acp_2014.png


### Conclusion

Branche au meilleur rang moyen (avec ou sans ACP), positionnement CEMAC.


In [48]:
if pd.isna(meilleure_globale['seuil_variance']):
    X_final = X
    print(f"Branche retenue : sans ACP -- {meilleure_globale['methode']} k={int(meilleure_globale['k'])}")
else:
    pca_finale = PCA(n_components=meilleure_globale['seuil_variance'], svd_solver='full', random_state=RNG_SEED)
    X_final = pca_finale.fit_transform(X)
    print(f"Branche retenue : ACP {meilleure_globale['seuil_variance']:.0%} "
          f"({X_final.shape[1]} composantes) -- {meilleure_globale['methode']} k={int(meilleure_globale['k'])}")

labels_finaux_acp = METHODES[meilleure_globale['methode']](X_final, int(meilleure_globale['k']))

df_assignation_finale = pd.DataFrame({'COUNTRY_CODE': pays_index, 'CLUSTER': labels_finaux_acp})
df_assignation_finale['PAYS_CEMAC'] = df_assignation_finale['COUNTRY_CODE'].isin(CEMAC)

chemin_assignation_finale = BASE / 'data' / f'CLUSTERING_ASSIGNATION_RETENUE_{ANNEE_ETUDE}.csv'
df_assignation_finale.to_csv(chemin_assignation_finale, index=False)
print(f"Taille des clusters : {df_assignation_finale['CLUSTER'].value_counts().sort_index().to_dict()}")
print(f"Assignation finale -> {chemin_assignation_finale}")

print("\nPositionnement des pays CEMAC (configuration retenue) :")
for code_pays, nom in CEMAC.items():
    ligne = df_assignation_finale[df_assignation_finale['COUNTRY_CODE'] == code_pays]
    if ligne.empty:
        print(f"  {nom} ({code_pays}) : absent du jeu de donnees {ANNEE_ETUDE}")
        continue
    cluster = ligne['CLUSTER'].iloc[0]
    pairs = df_assignation_finale[
        (df_assignation_finale['CLUSTER'] == cluster) & (~df_assignation_finale['PAYS_CEMAC'])
    ]['COUNTRY_CODE'].tolist()
    print(f"  {nom} ({code_pays}) -> cluster {cluster}, partage avec : {pairs}")

display(df_assignation_finale.sort_values(['CLUSTER', 'COUNTRY_CODE']))


Branche retenue : ACP 95% (1 composantes) -- k-means++ k=5
Taille des clusters : {0: 15, 1: 1, 2: 1, 3: 1, 4: 1}
Assignation finale -> m:\Travail\BEAC\balance_de_paiement\data\CLUSTERING_ASSIGNATION_RETENUE_2014.csv

Positionnement des pays CEMAC (configuration retenue) :
  Cameroun (CMR) -> cluster 4, partage avec : []
  Gabon (GAB) : absent du jeu de donnees 2014
  Republique du Congo (COG) -> cluster 0, partage avec : ['AGO', 'ALB', 'ARG', 'AZE', 'BOL', 'COL', 'GHA', 'ITA', 'LBY', 'NLD', 'PER', 'RUS', 'SDN', 'TTO']
  Tchad (TCD) : absent du jeu de donnees 2014
  Republique centrafricaine (CAF) : absent du jeu de donnees 2014
  Guinee equatoriale (GNQ) : absent du jeu de donnees 2014


,COUNTRY_CODE,CLUSTER,PAYS_CEMAC
0,AGO,0,False
1,ALB,0,False
2,ARG,0,False
3,AZE,0,False
4,BOL,0,False
6,COG,0,True
7,COL,0,False
9,GHA,0,False
10,ITA,0,False
12,LBY,0,False
